# Labeling the dataset by an transformer model
In this notebook, you are going to use the pre-trained transformer model, which is fine-tuned by the `transfomers_model_finetuner.py` script, to label the `labeled.csv` dataset. The first labeling method only labels a random sample of 700 unlabelled data points, while the second function labels the entire dataset.
## Pre-setup
### Loading the Model

In [1]:
import pandas as pd
import os

# The integrated AMD Radeon 840M is gfx1150 (RDNA 3.5), which the ROCm build of
# PyTorch has no precompiled kernels for. Presenting it as gfx1100 lets it run on
# the gfx1100 kernels. Must be set before torch is imported. No-op on non-ROCm.
os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

import random

/home/martan/Nienke/Protest_Labelling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_dir = "../models/hf_transformer_enhanced/hf_transformer_model"

In [3]:
print(f"\n--- Loading the model from {model_dir} ---")
loaded_tokenizer = AutoTokenizer.from_pretrained(model_dir)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_dir)

# Mirror transformers_model_finetuner.py: prefer MPS (Apple Silicon), then CUDA
# (the ROCm build of PyTorch exposes the AMD APU through the cuda API), then CPU.
if torch.backends.mps.is_available():
    load_device = torch.device("mps")
elif torch.cuda.is_available():
    load_device = torch.device("cuda")
else:
    load_device = torch.device("cpu")

loaded_model.to(load_device)

print(f"Using device: {load_device}")
print("Model and tokenizer loaded successfully!")
print("Model architecture:", loaded_model)


--- Loading the model from ../models/hf_transformer_enhanced/hf_transformer_model ---


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10669.29it/s]


Using device: cuda
Model and tokenizer loaded successfully!
Model architecture: BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_

### Loading the Dataset

In [4]:
dataset = pd.read_csv('../../data/labeled.csv')
dataset = dataset[dataset['class'] == 'unknown']
len(dataset)

/tmp/ipykernel_1738172/3490284572.py:1: DtypeWarning: Columns (0: civilian_targeting) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv('../../data/labeled.csv')


83921

In [5]:
# The saved model only stores generic LABEL_0..N names, so we must rebuild the
# index->class-name mapping the *exact* same way the finetuner did, or every
# prediction gets mistranslated (high val accuracy, garbage labels in practice).
#
# The finetuner (transformers_model_finetuner.py / load_data) does:
#   df = df[df['class'] != 'NoN']
#   class_names = sorted(df['class'].unique())
#   index -> name = {i+1: name for i, name in enumerate(class_names)}
# and predicts with argmax(logits)+1. We reproduce that here from the SAME
# training CSV, then assert the class count matches the loaded model's output
# dimension so a model/data mismatch fails loudly instead of silently.

TRAIN_CSV = '../../data/labeled_balanced.csv'  # the CSV the model was trained on

_train_df = pd.read_csv(TRAIN_CSV)
_train_df = _train_df[_train_df['class'] != 'NoN']
_class_names = sorted(_train_df['class'].unique())

Index_to_class = {i + 1: name for i, name in enumerate(_class_names)}
class_to_index = {v: k for k, v in Index_to_class.items()}

_n_model_classes = loaded_model.config.num_labels
assert len(Index_to_class) == _n_model_classes, (
    f"Label mismatch: '{TRAIN_CSV}' yields {len(Index_to_class)} classes "
    f"{_class_names}, but the loaded model outputs {_n_model_classes}. "
    f"This model was not trained on this CSV — point TRAIN_CSV at the exact "
    f"file used to train '{model_dir}'."
)

print(f"Reconstructed {len(Index_to_class)} labels from {TRAIN_CSV}:")
for i, name in Index_to_class.items():
    print(f"  {i}: {name}")

Reconstructed 15 labels from ../../data/labeled_balanced.csv:
  1: Black Rights
  2: Climate Action & Animal Welfare
  3: Climate Action & Resource Management
  4: Culture
  5: Education
  6: Justice & Civil Rights
  7: Labor Rights
  8: Palestine-Israel Conflict
  9: Public Services & Social Welfare
  10: Ukraine-Russia War
  11: environment
  12: lgbtq
  13: pandemic
  14: policies & politics
  15: unknown


## Labeling on 700 unlabeled data

In [6]:
unkown_data = []
random_indexes = random.sample(range(len(dataset)), 700)

Change the last line to save it to another file

In [7]:
MAX_LEN_HF = 128
for i in random_indexes:
    text = dataset['clean_notes'].iloc[i]
    onfiltert_text = dataset['notes'].iloc[i]
    inputs = loaded_tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN_HF)
    inputs = {key: val.to(load_device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = loaded_model(**inputs)
        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=1).item() + 1  

    unkown_data.append([onfiltert_text, Index_to_class[predicted_class], dataset['class'].iloc[i]])

new_df = pd.DataFrame(unkown_data, columns=['notes', 'class', 'orginal_class'])
new_df.to_csv('../../data/KINBERT_3000.csv', index=False)

/home/martan/Nienke/Protest_Labelling/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:92: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:287.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/martan/Nienke/Protest_Labelling/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:92: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:338.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


## Labeling over the hole dataset
Change the last line to save it to another file

In [ ]:
MAX_LEN_HF = 128

# NOTE: `dataset` was filtered (dataset[dataset['class'] == 'unknown']), so its
# index is NOT 0..N-1 — it keeps the original row labels and is full of gaps.
# We therefore read positionally with .iloc and collect predictions into a list,
# then assign that list as a column (list assignment aligns by POSITION, so the
# gappy index is irrelevant). The previous version wrote with `dataset.at[i, ...]`
# (LABEL-based), which scattered predictions onto the wrong rows and enlarged the
# frame with all-NaN rows whenever label `i` didn't exist.
predictions = []

for i in tqdm(range(len(dataset)), desc="Labeling dataset"):
    text = dataset['clean_notes'].iloc[i]
    inputs = loaded_tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN_HF)
    inputs = {key: val.to(load_device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = loaded_model(**inputs)
        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=1).item() + 1

    predictions.append(Index_to_class[predicted_class])

dataset['predicted_class'] = predictions
dataset.to_csv('../../data/filtered_events_class_with_predicted.csv', index=False)

In [ ]:
# ## Batched labeling (fast)
# Same result as the loop above, but ~1-2 orders of magnitude faster:
#   - Processes BATCH_SIZE texts per forward pass instead of one at a time.
#   - Sorts by length so each batch pads to its own longest sequence
#     (dynamic padding), wasting almost no compute on pad tokens. The original
#     order is restored before writing, so rows still line up with `dataset`.
#   - bf16 autocast on CUDA (mirrors the finetuner) for extra throughput.
#   - inference_mode (lighter than no_grad).
# As before, we read positionally and assign a list as a column, so the gappy
# index left by the `class == 'unknown'` filter doesn't matter.

MAX_LEN_HF = 128
BATCH_SIZE = 256  # lower this if you hit out-of-memory

texts = [str(t) for t in dataset['clean_notes'].tolist()]
n = len(texts)

# Sort indices by (proxy) length so similar-length texts batch together and
# dynamic padding stays tight. char length is a good, cheap proxy for token count.
order = sorted(range(n), key=lambda i: len(texts[i]))

predictions = [None] * n  # filled at original positions

loaded_model.eval()
use_bf16 = (load_device.type == 'cuda')

for start in tqdm(range(0, n, BATCH_SIZE), desc="Labeling dataset (batched)"):
    batch_pos = order[start:start + BATCH_SIZE]
    batch_texts = [texts[i] for i in batch_pos]

    inputs = loaded_tokenizer(
        batch_texts,
        return_tensors='pt',
        padding='longest',      # dynamic padding: pad to the longest in THIS batch
        truncation=True,
        max_length=MAX_LEN_HF,
    )
    inputs = {key: val.to(load_device) for key, val in inputs.items()}

    with torch.inference_mode():
        with torch.autocast(device_type=load_device.type, dtype=torch.bfloat16, enabled=use_bf16):
            logits = loaded_model(**inputs).logits
        predicted = (torch.argmax(logits, dim=1) + 1).cpu().tolist()

    for pos, cls_idx in zip(batch_pos, predicted):
        predictions[pos] = Index_to_class[cls_idx]

dataset['predicted_class'] = predictions
dataset.to_csv('../../data/filtered_events_class_with_predicted.csv', index=False)
